# 02 — Lemmy Moderation Log Collection

Collects public moderation logs from three Lemmy instances:
- **lemmy.ml** — tech-focused, strict moderation
- **lemmy.world** — general, largest instance
- **beehaw.org** — community-first, explicit moderation guidelines

Data is stored in `../data/moderation.db` tables:
- `lemmy_posts` — post content (title + body)
- `lemmy_modlog` — removal actions, reasons, moderator names

No authentication required — modlogs are public per Lemmy's design.

In [1]:
import sys
sys.path.insert(0, "..")

import sqlite3
import pandas as pd
from src.lemmy_scraper import get_conn, setup_db, INSTANCES

print("Target instances:", INSTANCES)

Target instances: ['https://lemmy.ml', 'https://lemmy.world', 'https://beehaw.org']


## 1. Connectivity check

In [2]:
import requests

for instance in INSTANCES:
    try:
        resp = requests.get(
            f"{instance}/api/v3/modlog",
            params={"limit": 1, "type_": "ModRemovePost"},
            headers={"Accept": "application/json"},
            timeout=10
        )
        print(f"  {instance:<30} → HTTP {resp.status_code}")
    except Exception as e:
        print(f"  {instance:<30} → ERROR: {e}")

  https://lemmy.ml               → HTTP 200
  https://lemmy.world            → HTTP 200
  https://beehaw.org             → HTTP 200


## 2. Database setup

In [3]:
setup_db()
print("✓ Schema ready")

✓ Schema ready


## 3. Collect moderation logs

In [4]:
from src.lemmy_scraper import collect_all_instances

total_posts, total_logs = collect_all_instances()
print(f"\nTotal posts stored   : {total_posts}")
print(f"Total modlog entries : {total_logs}")

11:00:05  INFO  https://lemmy.ml → 960 posts, 1000 modlog entries
11:00:50  INFO  https://lemmy.world → 962 posts, 1000 modlog entries
11:01:19  INFO  https://beehaw.org → 943 posts, 1000 modlog entries
11:01:19  INFO  Done — 2865 posts, 3000 modlog entries total



Total posts stored   : 2865
Total modlog entries : 3000


## 4. Row counts

In [5]:
conn = get_conn()
print("── Table sizes ──────────────────────────────")
for t in ["lemmy_posts", "lemmy_modlog"]:
    n = conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"  {t:<22}: {n:>6} rows")
conn.close()

── Table sizes ──────────────────────────────
  lemmy_posts           :   2865 rows
  lemmy_modlog          :   3000 rows


## 5. Sample modlog records

In [6]:
conn = get_conn()
df = pd.read_sql_query("""
    SELECT
        m.instance,
        SUBSTR(p.title, 1, 80) AS title_preview,
        m.mod_action,
        m.reason,
        m.removed,
        m.actioned_at
    FROM lemmy_modlog m
    LEFT JOIN lemmy_posts p ON m.post_id = p.post_id AND m.instance = p.instance
    WHERE m.reason IS NOT NULL AND m.reason != ''
    ORDER BY m.actioned_at DESC
    LIMIT 10
""", conn)
conn.close()

pd.set_option("display.max_colwidth", 85)
df

,instance,title_preview,mod_action,reason,removed,actioned_at
0,https://lemmy.ml,Fox Breaking News: Marine Jacked Off in 2005,mod_remove_post,not a meme,1,2026-05-29T14:47:24.292661Z
1,https://lemmy.world,Fox Breaking News: Marine Jacked Off in 2005,mod_remove_post,not a meme,1,2026-05-29T14:47:11.943288Z
2,https://lemmy.ml,The often forgotten or ignored part of the rainbow 🌈,mod_remove_post,not a meme,1,2026-05-29T14:45:46.759165Z
3,https://lemmy.world,The often forgotten or ignored part of the rainbow 🌈,mod_remove_post,not a meme,1,2026-05-29T14:45:40.683353Z
4,https://lemmy.ml,Ave Maria,mod_remove_post,AI slop,1,2026-05-29T14:44:39.752921Z
5,https://lemmy.world,Ave Maria,mod_remove_post,AI slop,1,2026-05-29T14:44:31.629046Z
6,https://lemmy.ml,"Roku OS's home screen now features a large, permanent ad",mod_remove_post,AM: Duplicate URL (Posted within last 3 days),1,2026-05-29T14:39:41.248263Z
7,https://lemmy.world,"Roku OS's home screen now features a large, permanent ad",mod_remove_post,AM: Duplicate URL (Posted within last 3 days),1,2026-05-29T14:39:30.854704Z
8,https://lemmy.ml,To my fellow Americans and my Haters,mod_remove_post,Rule 6,1,2026-05-29T14:36:11.393939Z
9,https://lemmy.ml,Found in the wild,mod_remove_post,ableism,1,2026-05-29T14:10:18.371338Z


## 6. Removal reason distribution

In [7]:
conn = get_conn()
df_reasons = pd.read_sql_query("""
    SELECT
        instance,
        reason,
        COUNT(*) AS count
    FROM lemmy_modlog
    WHERE reason IS NOT NULL AND reason != ''
    GROUP BY instance, reason
    ORDER BY count DESC
    LIMIT 20
""", conn)
conn.close()
print(df_reasons.to_string(index=False))

           instance                                        reason  count
 https://beehaw.org                   Outside the community scope     70
https://lemmy.world                        AutoMod rule triggered     67
   https://lemmy.ml                   Outside the community scope     53
 https://beehaw.org        Noncompliance with the post guidelines     49
   https://lemmy.ml                        AutoMod rule triggered     49
   https://lemmy.ml         Rule 10, account age is under 7 days.     46
https://lemmy.world                                        Rule 3     46
https://lemmy.world                   Outside the community scope     45
https://lemmy.world         Rule 10, account age is under 7 days.     41
https://lemmy.world                                        Rule 2     37
   https://lemmy.ml                                        Rule 3     35
   https://lemmy.ml                            AM: Violates Rules     32
https://lemmy.world                            AM: 